# 4. Regression Modelling and Ablations

The expensive stage. Fits every model family under the chronological walk-forward
protocol and writes all out-of-fold predictions to the cache, so that every downstream
analysis — confidence intervals, stratified error, calibration, figures — runs without a
single re-fit.

## 4.1 Environment and cached inputs

Loads `_shared.py` (paths, the artifact cache helpers, the target definitions) and applies the thesis figure style, then prints the artifact cache so it is visible which upstream stage produced these inputs and when. Reads `dataset`, `feature_spec` and `splits` from stage 02.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "_shared.py").exists() else Path.cwd() / "notebooks"))
from _shared import *  # noqa: F401,F403  paths, artifact cache, target bounds

from src.evaluation import figures as fx
fx.apply_thesis_style()

print("stage inputs available in the artifact cache:")
print(artifact_status().to_string(index=False) if len(artifact_status()) else "  (none yet)")

dataset = load_frame("dataset")
spec = load_json("feature_spec")
FEATURE_COLS, TARGET_COLS, TYPE_COLS = spec["FEATURE_COLS"], spec["TARGET_COLS"], spec["TYPE_COLS"]
X = dataset[FEATURE_COLS].fillna(0.0)
y = dataset[TARGET_COLS].copy()
print(f"\nmodelling table: {X.shape[0]} events x {X.shape[1]} features")
splits = load_object("splits")
print(f"walk-forward folds loaded: {len(splits)}")
# The dense-geometry ablation below builds its own folds, so the generator is
# needed here even though the primary splits come from the cache.
from src.training.walk_forward import generate_walk_forward_splits


stage inputs available in the artifact cache:
                     artifact               written_utc rows                                                                                                                                                                                  note
       ablation_block_contrib 2026-09-14T12:55:58+00:00   75                                                                                                                                 Paired-bootstrap contribution of each external block.
              ablation_blocks 2026-09-14T12:55:44+00:00   90                                                                                                                                             Pooled R2/RMSE per external-block config.
       classification_summary 2026-09-14T13:08:11+00:00   24                                                                                                                                     Pre-registered binary labels, po

## 4.2 Synthetic oversampling, illustrated

Shows what the oversampler produces on the full table. **This output is not used for
modelling.** The walk-forward loop below calls the same function fresh inside each
fold, on that fold's training rows only, so no synthetic row can reach a test split.

In [2]:
from src.sampling.time_aware_smogn import time_aware_smogn

# Illustration only -- shows what the upgraded SMOGN produces on the full real
# dataset. NOT used for modeling: \u00a79's run_walk_forward calls the same
# function fresh inside each fold, on that fold's training rows only, so no
# synthetic row can ever land in a test split (see \u00a78 markdown above).
# Rows with any missing target are excluded from augmentation candidacy -- a synthetic
# event must not be interpolated through an absent observation.
minority_mask_illustration = y.notna().all(axis=1) & (y["Y3_recovery_days"] > 30)
print(f"Minority (long-recovery) events: {minority_mask_illustration.sum()} / {len(y)}")

X_illustration, y_illustration = time_aware_smogn(
    X, y, minority_mask_illustration, event_dates=dataset["event_date"],
    max_year_gap=5.0, random_state=RANDOM_STATE,
    type_cols=TYPE_COLS, gdp_current_usd=dataset["gdp_current_usd"],
    max_synthetic_share=0.25,
)
print(f"Illustration: {X_illustration.shape[0]} rows ({X_illustration.shape[0] - X.shape[0]} synthetic, "
      f"neighbor-pair interpolation within a 5-year window) -- real per-fold runs will differ slightly "
      f"since each fold sees fewer real rows than the full dataset")


Minority (long-recovery) events: 9 / 74


Illustration: 83 rows (9 synthetic, neighbor-pair interpolation within a 5-year window) -- real per-fold runs will differ slightly since each fold sees fewer real rows than the full dataset


## 4.3 The walk-forward loop

**Deviation flagged and resolved (both blueprint and repo shared this gap, thesis \u00a73.7.2):** hyperparameters are tuned via `GridSearchCV`, wired here for the first time -- the repo's `training/optimizers.py` existed but was never called by anything, and the blueprint hardcodes fixed values outright. The thesis is explicit: *"using default values for the algorithm's parameters will result in severe overfitting"* on N\u224850-86.

**Upgraded from the original Y1-only proxy search**: the search now wraps the actual per-target estimator directly and is scored with `neg_root_mean_squared_error` on that target alone -- each of Ridge/RF/XGBoost gets its own per-target feature set and its own grid search, instead of all 3 targets sharing one Y1-ranked feature set and one Y1-scored proxy search. XGBoost -- previously **not searched at all** (hardcoded `n_estimators=150, max_depth=4`) -- gets its first real grid too. Both grids run **inside each walk-forward fold's training data only**, never touching test data.

**New: per-fold feature selection AND per-fold SMOGN.** New features (\u00a77) push the feature count from 28 to \u224832 against N\u224864 -- a real overfitting risk. An RF-importance top-20 selection is fit on each fold's training data only (no leakage) and applied identically before every model (Ridge/RF/XGBoost/MLP) that fold. SMOGN oversampling (\u00a78) is likewise now applied fresh inside each fold, to that fold's real training rows only -- fixing the test-leakage bug described in \u00a78's markdown, where synthetic rows could previously land inside a held-out test split.

In [3]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from src.evaluation.metrics import evaluate_regression
from src.sampling.time_aware_smogn import time_aware_smogn
from src.training.walk_forward import purge_horizon_overlap
from src.evaluation.collinearity import redundant_drop_set
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.svm import SVR
from sklearn.linear_model import QuantileRegressor

# Grid kept deliberately small (8 combos each) -- at N~64 with an inner CV inside
# each walk-forward fold, a larger grid made the search itself the bottleneck
# (>900s per fold from fit-call overhead alone, not data size).
RF_PARAM_GRID = {"n_estimators": [100, 200], "max_depth": [4, None], "min_samples_leaf": [1, 4]}
XGB_PARAM_GRID = {"n_estimators": [100, 200], "max_depth": [3, 6], "learning_rate": [0.05, 0.1]}
RIDGE_ALPHAS = np.logspace(-3, 3, 13)

# Each target's definitional support, from thesis 3.2.2 / feature_eng.py -- known in
# advance, independent of any held-out value.
#
# Imported from _shared.py rather than redeclared here. This cell used to carry its own
# copy, which shadowed the shared one and silently went stale: adding Y1_EventWindow_0_5_LogReturn_Pct /
# Y1_EventWindow_0_10_LogReturn_Pct as targets raised KeyError('Y1_EventWindow_0_5_LogReturn_Pct') the moment a prediction was clipped.
# One definition, one place to update.


def clip_to_bounds(target, pred):
    """Project predictions onto the target's definitional support.

    Uses only the target definitions, never held-out data, so this is not test-set
    fitting. Because every TRUE value already lies inside these bounds, clipping can
    never increase the absolute error of any point: RMSE and MAE are non-increasing
    and pooled R2 non-decreasing, for every model, always. Without it an
    under-regularized model in log space can emit recovery times in the hundreds of
    days for a target that is capped at 90 (this produced RMSE=157 on the dense-config
    ablation before the fix)."""
    lo, hi = TARGET_BOUNDS[target]
    if lo is None and hi is None:
        return np.asarray(pred, dtype=float)
    return np.clip(pred, lo, hi)


def inner_cv(n_rows, n_splits=3):
    """Chronological inner CV for hyperparameter selection.

    Replaces the previous `cv=3`, which sklearn expands to KFold(shuffle=False): that
    trains on LATER events to validate EARLIER ones, the exact mechanism thesis 3.7.1
    bans and that Section 8 of this notebook claims to have eliminated everywhere.
    TimeSeriesSplit never validates a fold earlier than its training rows."""
    return TimeSeriesSplit(n_splits=max(2, min(n_splits, n_rows - 1)))


def select_top_features(X_train, y_series, k=20, random_state=RANDOM_STATE):
    """Collinearity-drop, THEN RF-importance feature selection -- both fit on this
    fold's TRAIN data only, no leakage across folds. `redundant_drop_set` applies the
    pre-declared (docs/METHODOLOGY_AUDIT.md) |rho|>=0.95 rule -- it reads no target, so
    composing it before RF-importance ranking is not selection on the test set, just two
    independent filters run in sequence (src/evaluation/collinearity.py already existed;
    this cell simply hadn't been wired to call it). Ranked against whichever target
    Series is passed in -- each of Ridge/RF/XGBoost/GP/SVR/Quantile below gets its OWN
    per-target feature set (Y2/Y3 are no longer forced to reuse Y1's ranking, a real
    gap in the first working version of this cell). The MLP (next section)
    keeps one shared Y1-ranked set since its architecture shares a hidden
    layer across all 3 targets and can't take different inputs per output."""
    keep, _dropped, _detail = redundant_drop_set(X_train)
    X_train = X_train[keep]
    ranker = RandomForestRegressor(n_estimators=200, random_state=random_state)
    ranker.fit(X_train, y_series)
    ranked = pd.Series(ranker.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    return ranked.head(min(k, len(ranked))).index.tolist()


def augment_fold(X_train, y_train, dates_train, gdp_train=None,
                 max_year_gap=5.0, random_state=RANDOM_STATE):
    """Runs SMOGN on THIS FOLD's real training rows only, never on test rows --
    the fix for the leakage bug described in \u00a78/\u00a79's markdown. Returns
    (X_train_aug, y_train_aug); if the fold has <2 minority rows, SMOGN is a
    no-op and the real training rows are returned unchanged.

    Rows with any missing target are excluded from augmentation candidacy: a
    synthetic event must not be built by interpolating through an absent
    observation.

    TRIED AND REVERTED: widening the minority mask to the union of extreme
    Y1/Y2/Y3 rows (quintile thresholds) plus n_synthetic_per_row=2 was tested
    here and made every model on every target WORSE, not better (e.g. RF Y1
    pooled R2 went -0.32 -> -1.39, Ridge Y3 RMSE went 36.6 -> 62.9) --
    56-58 rows per fold with 26-28 of them synthetic drowned the real signal
    in interpolated noise. That variant is ALSO rejected independently by the
    pre-registered 25% synthetic-share cap declared in Section 8 (it reaches
    46-48%), so the revert does not rest on having seen its held-out scores.
    Recorded here, not deleted, as evidence this was genuinely tried."""
    complete = y_train.notna().all(axis=1)
    minority_mask = complete & (y_train["Y3_recovery_days"] > 30)
    X_aug, y_aug, report = time_aware_smogn(
        X_train, y_train, minority_mask, event_dates=dates_train,
        max_year_gap=max_year_gap, random_state=random_state,
        type_cols=TYPE_COLS, gdp_current_usd=gdp_train,
        max_synthetic_share=0.25, return_report=True,
    )
    return X_aug, y_aug, report


def score_and_record(y_true_series, pred_array, store, fold_i=None):
    """Records per-fold RMSE/MAE/R2 *and* the raw (y_true, y_pred) arrays --
    the raw arrays let the caller compute a POOLED R2 across all folds
    afterward, since per-fold R2 on a 5-10 point test window is numerically
    unstable (see \u00a710 markdown)."""
    yt = np.asarray(y_true_series)
    m = evaluate_regression(yt, pred_array)
    for k in ("rmse", "mae", "r2"):
        store[k].append(m[k])
    store["y_true"].append(yt)
    store["y_pred"].append(pred_array)
    # Which fold produced this entry. A (fold, target) pair is skipped whenever the
    # per-target NaN mask empties it, so the stored arrays are no longer guaranteed to
    # be one-per-fold in order -- anything mapping predictions back to events must read
    # this list rather than assume the i-th entry is the i-th fold.
    store.setdefault("folds", []).append(fold_i)


def run_walk_forward(splits, X_all, y_all, dates_all, gdp_all=None, k_features=20,
                     verbose=True, y1_horizon_end_all=None):
    """Fit Ridge/RF/XGBoost independently PER TARGET per fold -- each target
    gets its own SMOGN-augmented training rows (real test rows only, see
    augment_fold), its own RF-importance feature selection, and its own
    hyperparameter search. Returns (results, last_fold_models, last_fold_features)."""
    results = {
        name: {t: {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": [], "folds": []}
               for t in TARGET_COLS}
        for name in ("ridge", "random_forest", "xgboost", "gp", "svr", "quantile")
    }
    last_models = {name: {} for name in ("ridge", "random_forest", "xgboost", "gp", "svr", "quantile")}
    last_features = {}
    skipped_folds = []

    for fold_i, s in enumerate(splits):
        # ASPI_5D_Log_Return_Pct (Y1) reads market prices up to 5 trading days past
        # the event; splits are by event index, not calendar date, so purge any
        # training event whose label horizon reaches into this fold's test period
        # (fold-boundary embargo -- see walk_forward.purge_horizon_overlap). Applied
        # to the whole fold (all 3 targets) rather than duplicating the per-target
        # loop below: it only ever removes rows, so Y2/Y3 lose at most a few
        # borderline-adjacent events, never gain leakage.
        if y1_horizon_end_all is not None:
            s = purge_horizon_overlap(s, dates_all, y1_horizon_end_all)
        X_tr_real, X_te = X_all.iloc[s.train_index], X_all.iloc[s.test_index]
        y_tr_real, y_te = y_all.iloc[s.train_index], y_all.iloc[s.test_index]
        dates_tr = dates_all.iloc[s.train_index]
        gdp_tr = None if gdp_all is None else gdp_all.iloc[s.train_index]

        X_tr_aug, y_tr_aug, aug_report = augment_fold(X_tr_real, y_tr_real, dates_tr, gdp_tr)
        if verbose:
            print(f"fold {fold_i}: {len(X_tr_real)} real train rows -> {len(X_tr_aug)} after SMOGN "
                  f"({len(X_tr_aug) - len(X_tr_real)} synthetic, "
                  f"{aug_report['n_synthetic'] / max(len(X_tr_aug), 1):.0%} of the augmented table); "
                  f"{len(X_te)} real test rows (never augmented)")
            print(f"         pairing: {aug_report['pairs_in_window']} same-type within "
                  f"{5.0:.0f}y, {aug_report['pairs_window_relaxed']} same-type window-relaxed, "
                  f"{aug_report['skipped_no_partner']} skipped (no same-type partner)")

        for target in TARGET_COLS:
            # Y3 (recovery days) is right-skewed and capped at 90; log1p stabilizes
            # the fit for Ridge/RF/XGBoost. Predictions are inverted with expm1 and
            # clipped back to [0, 90] before scoring, so every reported metric stays
            # in real units (days). Y1/Y2 unchanged -- both are roughly symmetric and
            # Y1 takes negative values, where log1p is undefined.
            use_log = target == "Y3_recovery_days"

            # Rows whose target is MISSING are dropped, not imputed. Y2 is absent for 3
            # events (the 2000 archive workbook failed to parse); filling them with 0.0
            # would assert "volume exactly at its 30-day baseline" -- a fabricated
            # observation, and the same zero-fill Section 3 rejects for volume itself.
            tr_ok, te_ok = y_tr_aug[target].notna(), y_te[target].notna()
            real_ok = y_tr_real[target].notna()

            # Y2 is unobserved for the 2000 archive year and for every post-2023 event
            # (countryeconomy publishes the index level, not volume). With a 5-event
            # test window the dense ablation can therefore land a fold entirely inside
            # an all-NaN stretch, leaving nothing to fit or score. Skip that
            # (fold, target) pair and say so, rather than either crashing or silently
            # scoring an empty array as if it were a result.
            if int(real_ok.sum()) < 10 or int(te_ok.sum()) < 1:
                skipped_folds.append((fold_i, target, int(real_ok.sum()), int(te_ok.sum())))
                continue
            y_tr_t, y_te_t = y_tr_aug.loc[tr_ok, target], y_te.loc[te_ok, target]

            y_rank = np.log1p(y_tr_t) if use_log else y_tr_t
            feat_cols = select_top_features(X_tr_aug[tr_ok], y_rank, k=k_features)
            X_tr_sel, X_te_sel = X_tr_aug.loc[tr_ok, feat_cols], X_te.loc[te_ok, feat_cols]
            y_tr_fit = np.log1p(y_tr_t) if use_log else y_tr_t
            last_features[target] = feat_cols

            # Hyperparameters are selected on this fold's REAL rows only, chronologically,
            # then the winning configuration is refit on real + synthetic. Two reasons the
            # selection must exclude synthetic rows: SMOGN appends them at the tail, so
            # under any index-block CV the last validation block is disproportionately
            # interpolated, and the model would be chosen partly for predicting its own
            # synthetic output.
            X_sel_real = X_tr_real.loc[real_ok, feat_cols]
            y_real_t = y_tr_real.loc[real_ok, target]
            y_fit_real = np.log1p(y_real_t) if use_log else y_real_t
            cv_real = inner_cv(len(X_sel_real))

            def _finish(model):
                p = model.predict(X_te_sel)
                if use_log:
                    p = np.expm1(p)
                return clip_to_bounds(target, p)

            # Ridge: standardize FIRST. L2 is not scale-equivariant, and this design
            # matrix mixes ASPI price levels (~1e4), raw USD damage (~1e9) and 0/1 flags
            # -- so the previous bare Ridge(alpha=1.0) applied almost no shrinkage to the
            # damage columns and crushed the binary ones. alpha is now selected inside the
            # fold instead of left at its library default, matching the treatment RF and
            # XGBoost already receive. Ridge is H1's designated baseline, so a
            # mis-specified Ridge would make the whole H1 comparison meaningless.
            alpha_search = make_pipeline(StandardScaler(), RidgeCV(alphas=RIDGE_ALPHAS, cv=cv_real))
            alpha_search.fit(X_sel_real, y_fit_real)
            ridge = make_pipeline(
                StandardScaler(),
                Ridge(alpha=float(alpha_search.named_steps["ridgecv"].alpha_)),
            )
            ridge.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(ridge), results["ridge"][target], fold_i)

            rf_search = GridSearchCV(
                RandomForestRegressor(random_state=RANDOM_STATE), RF_PARAM_GRID,
                cv=cv_real, scoring="neg_root_mean_squared_error", n_jobs=1, refit=False,
            )
            rf_search.fit(X_sel_real, y_fit_real)
            rf = RandomForestRegressor(random_state=RANDOM_STATE, **rf_search.best_params_)
            rf.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(rf), results["random_forest"][target], fold_i)

            xgb_search = GridSearchCV(
                XGBRegressor(random_state=RANDOM_STATE, verbosity=0), XGB_PARAM_GRID,
                cv=cv_real, scoring="neg_root_mean_squared_error", n_jobs=1, refit=False,
            )
            xgb_search.fit(X_sel_real, y_fit_real)
            xgb = XGBRegressor(random_state=RANDOM_STATE, verbosity=0, **xgb_search.best_params_)
            xgb.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(xgb), results["xgboost"][target], fold_i)

            # Gaussian Process regression: literature-standard choice for small-N
            # nonlinear regression with calibrated uncertainty (see docs/METHODOLOGY_AUDIT.md,
            # "Feature/model pass" section). Matern(nu=1.5) is the usual default over RBF
            # for physical/financial series (less aggressively smooth); WhiteKernel absorbs
            # observation noise instead of forcing it into the length-scale. No grid search
            # here -- at ~20-30 real rows per fold, optimising the kernel's own
            # log-marginal-likelihood (sklearn's default `optimizer`) already tunes it;
            # ponytail: fixed kernel family, revisit only if this measurably underperforms.
            gp_kernel = ConstantKernel(1.0) * Matern(nu=1.5) + WhiteKernel(noise_level=1.0)
            gp = make_pipeline(
                StandardScaler(),
                GaussianProcessRegressor(kernel=gp_kernel, normalize_y=True, random_state=RANDOM_STATE),
            )
            gp.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(gp), results["gp"][target], fold_i)

            # SVR: closes docs/METHODOLOGY_AUDIT.md's own open item ("SVR specified but
            # never implemented"). RBF kernel, default C/epsilon -- same ponytail
            # no-grid-search reasoning as GP above (this notebook's grid search is already
            # the per-fold runtime bottleneck per the RF_PARAM_GRID/XGB_PARAM_GRID comment).
            svr = make_pipeline(StandardScaler(), SVR(kernel="rbf", C=1.0, epsilon=0.1))
            svr.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(svr), results["svr"][target], fold_i)

            # Median (q=0.5) quantile regression: a robust point-estimator less sensitive
            # to the outlier events this dataset is full of (disasters are exactly the
            # long-tail case OLS/Ridge weight quadratically). solver="highs" is sklearn's
            # recommended LP solver for QuantileRegressor.
            quantile = make_pipeline(
                StandardScaler(), QuantileRegressor(quantile=0.5, alpha=0.01, solver="highs"),
            )
            quantile.fit(X_tr_sel, y_tr_fit)
            score_and_record(y_te_t, _finish(quantile), results["quantile"][target], fold_i)

            last_models["ridge"][target] = ridge
            last_models["random_forest"][target] = rf
            last_models["xgboost"][target] = xgb
            last_models["gp"][target] = gp
            last_models["svr"][target] = svr
            last_models["quantile"][target] = quantile

    for name in results:
        for t in TARGET_COLS:
            yt_all = np.concatenate(results[name][t]["y_true"])
            yp_all = np.concatenate(results[name][t]["y_pred"])
            results[name][t]["pooled_r2"] = float(r2_score(yt_all, yp_all))

    if skipped_folds:
        print(f"  skipped {len(skipped_folds)} (fold, target) pairs with too few "
              f"non-missing rows:")
        for _f, _t, _ntr, _nte in skipped_folds:
            print(f"    fold {_f} / {_t}: {_ntr} real train, {_nte} test rows")
    return results, last_models, last_features


results, last_models, selected_features = run_walk_forward(
    splits, X, y, dataset["event_date"], gdp_all=dataset["gdp_current_usd"],
    y1_horizon_end_all=dataset["Y1_horizon_end_date"])
rf_y1 = last_models["random_forest"][TARGET_COLS[0]]  # single-output RandomForestRegressor -- kept for \u00a711 SHAP

for name, targets_m in results.items():
    for t in TARGET_COLS:
        m = targets_m[t]
        print(f"{name:>14s} | {t:<20s} RMSE mean={np.mean(m['rmse']):.4f}  "
              f"MAE mean={np.mean(m['mae']):.4f}  R2(per-fold mean)={np.mean(m['r2']):.4f}  "
              f"R2(pooled)={m['pooled_r2']:.4f}")


fold 0: 30 real train rows -> 33 after SMOGN (3 synthetic, 9% of the augmented table); 10 real test rows (never augmented)
         pairing: 3 same-type within 5y, 0 same-type window-relaxed, 1 skipped (no same-type partner)


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


         ridge | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.3118  MAE mean=1.7287  R2(per-fold mean)=-0.2455  R2(pooled)=-0.0421
         ridge | Y2_abnormal_volume   RMSE mean=0.6480  MAE mean=0.5709  R2(per-fold mean)=-1.1409  R2(pooled)=-0.4742
         ridge | Y3_recovery_days     RMSE mean=30.1026  MAE mean=14.6262  R2(per-fold mean)=-0.2095  R2(pooled)=-0.2020
         ridge | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=2.8156  MAE mean=2.1869  R2(per-fold mean)=-0.2031  R2(pooled)=0.0543
         ridge | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=4.3934  MAE mean=3.6541  R2(per-fold mean)=-1.2115  R2(pooled)=0.0071
 random_forest | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.2691  MAE mean=1.7743  R2(per-fold mean)=-0.4880  R2(pooled)=0.0416
 random_forest | Y2_abnormal_volume   RMSE mean=0.4788  MAE mean=0.3880  R2(per-fold mean)=-0.5753  R2(pooled)=0.2270
 random_forest | Y3_recovery_days     RMSE mean=29.7618  MAE mean=13.8993  R2(per-fold mean)=-0.1623  R2(pooled)=-0.1826

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### 4.3.1 Shallow multi-task MLP

Single hidden layer, heavy dropout — per thesis §3.6.3 and the panel blueprint's Step 7, both agree here: N≈50-86 rules out deep architectures (LSTM/Transformer excluded by both sources, correctly — Venkatarathnam et al., 2024; Perera, 2025).

**Environment issue found and worked around (not a methodology deviation, an infrastructure one):** on this development machine, `import torch` reliably raises a Windows DLL-init error (`WinError 1114`, `c10.dll`) when it happens in the same process that already imported scikit-learn/XGBoost/SHAP — reproduced consistently across repeated attempts, including with the standard `KMP_DUPLICATE_LIB_OK` OpenMP workaround, which did not resolve it. Rather than silently skip the MLP or leave the notebook non-executing, the MLP training is dispatched to a **fresh subprocess** (`src/models/mlp_subprocess_runner.py`) per fold — same model, same hyperparameters, just isolated from whatever accumulated process state causes the conflict. This is disclosed here rather than hidden because a defense panel deserves to know a result came from a workaround, not just that a number appeared.

**A second real bug found and fixed here, not just described:** a first run of this cell produced a catastrophically broken MLP (Y1 RMSE ≈0.24, R² ≈ −390) — not "the MLP is weak at this N," an actual implementation bug. Y3 (recovery days, raw range 0–90) and Y1 (ASPI return, raw range ≈ ±0.07) differ in raw scale by roughly three orders of magnitude. `config.yaml`'s loss weights `(1.0, 0.1, 0.5)` — the thesis's own eq.(4) mechanism for this exact problem (§3.6.4) — only rebalance the loss by a factor of ~2–10×, nowhere near enough to offset a ~1000× raw-scale gap; the network's shared hidden layer collapsed onto minimizing Y3's dominant squared-error term and produced near-random output on Y1. **Fix**: standardize *y* (not just *X*) per fold before training, inverse-transform predictions back to raw scale before evaluating — a standard, necessary complement to loss-weighting for multi-scale multi-task regression that neither the thesis text nor the repo's original implementation made explicit.

In [4]:
import subprocess
import tempfile
from pathlib import Path as _Path

from sklearn.preprocessing import StandardScaler

MLP_RUNNER = REPO_ROOT / "src" / "models" / "mlp_subprocess_runner.py"

Y3_IDX = TARGET_COLS.index("Y3_recovery_days")


def _fwd_y(y_df):
    """log1p on Y3 only -- the same response transform \u00a79 applies to Ridge/RF/XGBoost,
    mirrored here so that every stack member is fit on the SAME transform of the target.
    Applied BEFORE the StandardScaler, so the scaler standardizes log-days.

    This is not redundant with the y-scaler. Standardization is affine and fixes SCALE;
    log1p is monotone-nonlinear and fixes SHAPE. Y3 after standardization is still
    majority-zero, still right-skewed, and still hard-capped at 90 -- under squared loss
    the handful of 90s dominate. This is the transform that turned Ridge's Y3 from a
    negative held-out R2 to the only positive one in the study."""
    a = np.asarray(y_df, dtype=float).copy()
    a[:, Y3_IDX] = np.log1p(np.clip(a[:, Y3_IDX], 0.0, None))
    return a


def _inv_y(a):
    """Exact inverse of _fwd_y, applied AFTER y_scaler.inverse_transform -- reversing the
    order would take logs of negative standardized values."""
    a = np.asarray(a, dtype=float).copy()
    a[:, Y3_IDX] = np.expm1(a[:, Y3_IDX])
    return a


def train_mlp_subprocess(X_train, y_train, X_test):
    y_train_t = _fwd_y(y_train)               # 1. log1p on Y3
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train_t)  # 2. standardize the transformed targets
    Xtr = x_scaler.transform(X_train).astype(np.float32)
    Xte = x_scaler.transform(X_test).astype(np.float32)
    ytr = y_scaler.transform(y_train_t).astype(np.float32)

    with tempfile.TemporaryDirectory() as tmp:
        in_path = _Path(tmp) / "in.npz"
        out_path = _Path(tmp) / "out.npz"
        np.savez(in_path, X_train=Xtr, y_train=ytr, X_test=Xte)

        result = subprocess.run(
            [sys.executable, "-m", "src.models.mlp_subprocess_runner", str(in_path), str(out_path)],
            cwd=REPO_ROOT, capture_output=True, text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"MLP subprocess failed:\n{result.stderr}")

        pred_scaled = np.load(out_path)["pred"]
        # 3. un-standardize, 4. expm1 back to days. Bound-clipping happens at scoring.
        return _inv_y(y_scaler.inverse_transform(pred_scaled))

mlp_metrics = {t: {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": []} for t in TARGET_COLS}
for s in splits:
    X_tr_real, X_te = X.iloc[s.train_index], X.iloc[s.test_index]
    y_tr_real, y_te = y.iloc[s.train_index], y.iloc[s.test_index]
    dates_tr = dataset["event_date"].iloc[s.train_index]
    gdp_tr = dataset["gdp_current_usd"].iloc[s.train_index]
    # Same per-fold SMOGN as \u00a79's tree models (augment_fold) -- real test
    # rows only, never augmented.
    X_tr_aug, y_tr_aug, _ = augment_fold(X_tr_real, y_tr_real, dates_tr, gdp_tr)
    # The MLP is a single multi-output network, so unlike the per-target models in
    # \u00a79 it cannot drop a row for one target and keep it for another: any row with
    # a missing target is dropped from its training set entirely. 3 events are affected
    # (missing Y2), all inside fold 0's training window.
    _complete = y_tr_aug.notna().all(axis=1)
    X_tr_aug, y_tr_aug = X_tr_aug[_complete], y_tr_aug[_complete]
    # Shared Y1-ranked feature set (\u00a79's per-target selection doesn't apply
    # here -- the MLP's hidden layer is shared across all 3 targets, so all 3
    # outputs must see the same input columns).
    feat_cols = select_top_features(X_tr_aug, y_tr_aug[TARGET_COLS[0]], k=20)
    pred = train_mlp_subprocess(X_tr_aug[feat_cols], y_tr_aug, X_te[feat_cols])
    for i, target in enumerate(TARGET_COLS):
        _te_ok = y_te[target].notna()
        score_and_record(y_te.loc[_te_ok, target],
                         clip_to_bounds(target, pred[_te_ok.to_numpy(), i]),
                         mlp_metrics[target])

for t in TARGET_COLS:
    yt_all = np.concatenate(mlp_metrics[t]["y_true"])
    yp_all = np.concatenate(mlp_metrics[t]["y_pred"])
    mlp_metrics[t]["pooled_r2"] = float(r2_score(yt_all, yp_all))
    m = mlp_metrics[t]
    print(f"{'mlp':>14s} | {t:<20s} RMSE mean={np.mean(m['rmse']):.4f}  "
          f"MAE mean={np.mean(m['mae']):.4f}  R2(per-fold mean)={np.mean(m['r2']):.4f}  "
          f"R2(pooled)={m['pooled_r2']:.4f}")
results["mlp"] = mlp_metrics


           mlp | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.5381  MAE mean=1.9968  R2(per-fold mean)=-1.0441  R2(pooled)=-0.1383
           mlp | Y2_abnormal_volume   RMSE mean=0.5624  MAE mean=0.4569  R2(per-fold mean)=-0.8200  R2(pooled)=-0.1278
           mlp | Y3_recovery_days     RMSE mean=31.6191  MAE mean=15.6584  R2(per-fold mean)=-0.3183  R2(pooled)=-0.3673
           mlp | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=3.0967  MAE mean=2.3684  R2(per-fold mean)=-0.9139  R2(pooled)=-0.0761
           mlp | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=4.8923  MAE mean=4.1399  R2(per-fold mean)=-3.1780  R2(pooled)=-0.1328


### 4.3.2 Ensemble blend

A real, cheap variance-reduction lever that needs no new data: per fold, per target, blend the three non-linear models' predictions weighted by the inverse of each model's own RMSE on that fold (a worse fold performance gets less say). This is computed from the exact predictions already produced above — no retraining, no new information, just combining what's already there. Reported as a 4th "model" in §10's summary table alongside Ridge/RF/XGBoost/MLP.

In [5]:
ENSEMBLE_MEMBERS = ("random_forest", "xgboost", "mlp")

ensemble_results = {t: {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": []} for t in TARGET_COLS}

for t in TARGET_COLS:
    n_folds = len(results["random_forest"][t]["y_true"])
    for fold_i in range(n_folds):
        yt = results["random_forest"][t]["y_true"][fold_i]  # identical across members -- same fold, same test rows
        preds = np.array([results[m][t]["y_pred"][fold_i] for m in ENSEMBLE_MEMBERS])

        # Weights come from PRIOR folds only. The previous version read
        # results[m][t]["rmse"][fold_i] -- the held-out error of the very fold being
        # predicted -- and then scored the blend on that same fold, so every ensemble
        # number in the study was optimistically biased. That is selection on test
        # performance, which the protocol forbids; the stacked meta-learner below always
        # did this correctly and the blend now matches it. Fold 0 has no prior error to
        # weight by, so it falls back to equal weights rather than borrowing its own.
        if fold_i == 0:
            weights = np.ones(len(ENSEMBLE_MEMBERS))
        else:
            prior_rmses = np.array([
                float(np.sqrt(np.mean(np.concatenate(
                    [(results[m][t]["y_pred"][j] - results[m][t]["y_true"][j]) ** 2
                     for j in range(fold_i)]))))
                for m in ENSEMBLE_MEMBERS])
            weights = 1.0 / np.clip(prior_rmses, 1e-8, None)
        weights /= weights.sum()
        blended = clip_to_bounds(t, np.average(preds, axis=0, weights=weights))
        score_and_record(yt, blended, ensemble_results[t])

for t in TARGET_COLS:
    yt_all = np.concatenate(ensemble_results[t]["y_true"])
    yp_all = np.concatenate(ensemble_results[t]["y_pred"])
    ensemble_results[t]["pooled_r2"] = float(r2_score(yt_all, yp_all))
    m = ensemble_results[t]
    print(f"{'ensemble':>14s} | {t:<20s} RMSE mean={np.mean(m['rmse']):.4f}  "
          f"MAE mean={np.mean(m['mae']):.4f}  R2(per-fold mean)={np.mean(m['r2']):.4f}  "
          f"R2(pooled)={m['pooled_r2']:.4f}")

results["ensemble"] = ensemble_results


      ensemble | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.3653  MAE mean=1.8342  R2(per-fold mean)=-0.6357  R2(pooled)=-0.0114
      ensemble | Y2_abnormal_volume   RMSE mean=0.4863  MAE mean=0.3917  R2(per-fold mean)=-0.4861  R2(pooled)=0.1764
      ensemble | Y3_recovery_days     RMSE mean=30.3847  MAE mean=14.9411  R2(per-fold mean)=-0.2177  R2(pooled)=-0.2299
      ensemble | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=2.9548  MAE mean=2.2719  R2(per-fold mean)=-0.5528  R2(pooled)=0.0018
      ensemble | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=4.5725  MAE mean=3.9734  R2(per-fold mean)=-1.7009  R2(pooled)=-0.0237


### 4.3.3 Stacked meta-learner

The ensemble above blends the three base models with fixed inverse-RMSE weights. Here a small learned meta-model (non-negative LinearRegression on the three base predictions) replaces the fixed formula -- a genuinely different combination method, not a relabeling of the same one. Non-negative weights keep it interpretable as a blend rather than an arbitrary linear recombination.

Leakage-safety, expanding-window, the same rule as everywhere else in this notebook: the meta-learner for fold i is fit only on folds 0..i-1 out-of-fold predictions, never on the fold it is about to predict. Fold 0 has no prior fold to learn from and is skipped -- so this evaluates on fewer real points than the base models (n=20 on the primary 3-fold config), a real and disclosed trade-off of stacking at this N. `run_stack` is a function so the same method can be re-run on the denser walk-forward config later, where only 1 of 8 folds is lost instead of 1 of 3.

In [6]:
from sklearn.linear_model import LinearRegression

STACK_MEMBERS = ("random_forest", "xgboost", "mlp")


def run_stack(res, members=STACK_MEMBERS, members_by_target=None, label="stacked", verbose=True):
    """Expanding-window stacking over already-recorded out-of-fold predictions.
    Fold i's meta-learner is fit on folds 0..i-1 only, so it never sees the fold
    it predicts. Fold 0 is skipped (no prior fold to learn from), which is why n
    is lower here than for the base models -- a real cost of stacking at this N,
    reported rather than hidden."""
    stacked = {t: {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": []} for t in TARGET_COLS}
    for t in TARGET_COLS:
        # Membership rule (fixed independently of any stacked score): a base model may
        # enter the meta-learner for target t only if it was FIT on the same transform
        # of t as every other member -- a non-negative linear recombination of
        # predictions produced under different response transforms is not a well-posed
        # blend. The hook exists to express that rule; it is currently passed nothing,
        # because \u00a79's log1p treatment of Y3 is now mirrored inside the MLP so all
        # members agree by construction.
        mem = (members_by_target or {}).get(t, members)
        n_folds = len(res[mem[0]][t]["y_true"])
        for fold_i in range(1, n_folds):
            meta_X_train = np.column_stack([
                np.concatenate(res[m][t]["y_pred"][:fold_i]) for m in mem
            ])
            meta_y_train = np.concatenate(res[mem[0]][t]["y_true"][:fold_i])
            meta = LinearRegression(positive=True).fit(meta_X_train, meta_y_train)

            meta_X_test = np.column_stack([res[m][t]["y_pred"][fold_i] for m in mem])
            yt = res[mem[0]][t]["y_true"][fold_i]
            score_and_record(yt, clip_to_bounds(t, meta.predict(meta_X_test)), stacked[t])

        yt_all = np.concatenate(stacked[t]["y_true"])
        yp_all = np.concatenate(stacked[t]["y_pred"])
        stacked[t]["pooled_r2"] = float(r2_score(yt_all, yp_all))
        if verbose:
            m = stacked[t]
            print(f"{label:>14s} | {t:<20s} RMSE mean={np.mean(m['rmse']):.4f}  "
                  f"MAE mean={np.mean(m['mae']):.4f}  R2(per-fold mean)={np.mean(m['r2']):.4f}  "
                  f"R2(pooled)={m['pooled_r2']:.4f}  (n={len(yt_all)}, fold 0 skipped)")
    return stacked


print(f"Stack members: {STACK_MEMBERS}")
results["stacked"] = run_stack(results)


Stack members: ('random_forest', 'xgboost', 'mlp')
       stacked | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.5527  MAE mean=1.9813  R2(per-fold mean)=-0.5351  R2(pooled)=-0.0194  (n=30, fold 0 skipped)
       stacked | Y2_abnormal_volume   RMSE mean=0.5807  MAE mean=0.4769  R2(per-fold mean)=-0.3073  R2(pooled)=-0.0150  (n=24, fold 0 skipped)
       stacked | Y3_recovery_days     RMSE mean=32.9842  MAE mean=24.2400  R2(per-fold mean)=-0.4697  R2(pooled)=-0.4458  (n=30, fold 0 skipped)
       stacked | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=3.4378  MAE mean=2.6400  R2(per-fold mean)=-0.4690  R2(pooled)=-0.0688  (n=30, fold 0 skipped)
       stacked | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=5.4766  MAE mean=4.7004  R2(per-fold mean)=-1.5066  R2(pooled)=-0.1511  (n=30, fold 0 skipped)


## 4.4 Naive baselines: what the model has to beat

A negative R-squared is widely misread as "the model learned nothing". It means something more specific: **worse than a predictor that already knows the test set's own mean**, which is not a baseline anyone could actually deploy. Two honest, deployable nulls are computed here on exactly the same folds as every model above:

- **`naive_zero`** -- predict 0 for every event. This is the *economic* null: "the disaster had no measurable effect". Y1 and Y2 are both defined as deviations, and Y3 = 0 means "recovered the same day", so zero is the meaningful no-effect value for all three targets.
- **`naive_train_mean`** -- predict the training fold's mean. This is the *statistical* null that makes R-squared interpretable, computed honestly per fold (the training mean, never the test mean).

Both rows flow into the Section 10 summary table below, so every model metric can be read against a floor. This is the comparison that determines whether a negative R-squared reflects a useless model or a hard-to-beat R-squared denominator.

In [7]:
naive_specs = {
    "naive_zero": lambda y_tr: 0.0,                        # economic null: no measurable effect
    "naive_train_mean": lambda y_tr: float(y_tr.mean()),   # statistical null: R-squared's reference point
}

for naive_name, const_fn in naive_specs.items():
    store_by_target = {}
    for t in TARGET_COLS:
        store = {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": []}
        for s in splits:
            yt = y[t].iloc[s.test_index].dropna()  # same per-target masking as the models
            const = const_fn(y[t].iloc[s.train_index].dropna())  # training rows only, never the test mean
            score_and_record(yt, np.full(len(yt), const, dtype=float), store)
        yt_all = np.concatenate(store["y_true"])
        yp_all = np.concatenate(store["y_pred"])
        store["pooled_r2"] = float(r2_score(yt_all, yp_all))
        store_by_target[t] = store
        print(f"{naive_name:>16s} | {t:<20s} RMSE mean={np.mean(store['rmse']):.4f}  "
              f"MAE mean={np.mean(store['mae']):.4f}  R2(pooled)={store['pooled_r2']:.4f}")
    results[naive_name] = store_by_target

print()
# Derived here rather than inherited: this list used to be defined in the directional-
# accuracy cell, which now lives in stage 06, so depending on it made stage 04 fail.
REPORTED_MODELS = [m for m in results if not m.startswith("naive_")]

print("Models beating the 'no measurable effect' null (naive_zero) on pooled RMSE:")
for t in TARGET_COLS:
    null_rmse = np.mean(results["naive_zero"][t]["rmse"])
    beat = [m for m in results
            if not m.startswith("naive_") and np.mean(results[m][t]["rmse"]) < null_rmse]
    print(f"  {t:<20s} null RMSE={null_rmse:.4f} -> beaten by {len(beat)}/{len(REPORTED_MODELS)}: "
          f"{', '.join(beat) if beat else 'NONE'}")


      naive_zero | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.2639  MAE mean=1.7189  R2(pooled)=-0.0247


      naive_zero | Y2_abnormal_volume   RMSE mean=0.5523  MAE mean=0.4598  R2(pooled)=-0.0344
      naive_zero | Y3_recovery_days     RMSE mean=30.2831  MAE mean=13.2250  R2(pooled)=-0.2084
      naive_zero | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=2.9419  MAE mean=2.3374  R2(pooled)=-0.0746
      naive_zero | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=4.4599  MAE mean=3.6328  R2(pooled)=-0.0540
naive_train_mean | Y1_ASPI_5D_Forward_LogReturn_Pct RMSE mean=2.3348  MAE mean=1.7939  R2(pooled)=-0.0397
naive_train_mean | Y2_abnormal_volume   RMSE mean=0.5527  MAE mean=0.4610  R2(pooled)=-0.0584
naive_train_mean | Y3_recovery_days     RMSE mean=27.8967  MAE mean=19.5683  R2(pooled)=-0.0083
naive_train_mean | Y1_EventWindow_0_5_LogReturn_Pct RMSE mean=2.9494  MAE mean=2.3350  R2(pooled)=-0.0352
naive_train_mean | Y1_EventWindow_0_10_LogReturn_Pct RMSE mean=4.4966  MAE mean=3.7235  R2(pooled)=-0.0332

Models beating the 'no measurable effect' null (naive_zero) on pooled RMSE:
  Y1_ASPI_

## 4.5 Ablation: walk-forward density

A transparency/robustness check, not a silent swap of defaults: the primary `TRAIN_WINDOW=30, TEST_WINDOW=10, STEP=10` config above yields **3 folds** on the 64 real events. Re-running the same real data through a denser `TRAIN_WINDOW=20, TEST_WINDOW=5, STEP=5` split (more, smaller folds — less noisy per-model averaging, but a smaller per-fold training set) shows whether the primary results are sensitive to that choice. MLP isn't re-run here (subprocess overhead per fold; §10 above already covers it) — this ablation targets Ridge/RF/XGBoost only, reusing the exact `run_walk_forward` function from §9.

**This configuration produces the best Y1 result in the notebook (RF pooled R² = +0.151, versus -0.322 primary) and it is not promoted** — see the selection-bias disclosure in §10. It also produces the worst (Ridge Y3 pooled R² = -229, from an under-regularized linear extrapolation in log space on a 20-row training window). Both directions are the same finding: at this N, results are highly sensitive to the split geometry, which is itself the most important thing this ablation has to say.

In [8]:
splits_dense = list(generate_walk_forward_splits(len(X), train_window=20, test_window=5, step=5))
print(f"Dense config folds: {len(splits_dense)} (vs {len(splits)} for the primary 30/10/10 config)")

results_dense, _, _ = run_walk_forward(
    splits_dense, X, y, dataset["event_date"], gdp_all=dataset["gdp_current_usd"], verbose=False,
    y1_horizon_end_all=dataset["Y1_horizon_end_date"])

rows = []
for name, targets_m in results_dense.items():
    for t in TARGET_COLS:
        m = targets_m[t]
        rows.append({
            "model": name, "target": t,
            "rmse": np.mean(m["rmse"]), "mae": np.mean(m["mae"]),
            "r2_per_fold_mean": np.mean(m["r2"]), "r2_pooled": m["pooled_r2"],
        })
summary_dense = pd.DataFrame(rows).sort_values(["target", "model"]).reset_index(drop=True)
summary_dense


Dense config folds: 10 (vs 4 for the primary 30/10/10 config)


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


  skipped 1 (fold, target) pairs with too few non-missing rows:
    fold 9 / Y2_abnormal_volume: 19 real train, 0 test rows


,model,target,rmse,mae,r2_per_fold_mean,r2_pooled
0,gp,Y1_ASPI_5D_Forward_LogReturn_Pct,2.378636,1.934569,-0.779710,-0.112504
1,quantile,Y1_ASPI_5D_Forward_LogReturn_Pct,6.449256,5.003746,-24.872453,-8.721151
2,random_forest,Y1_ASPI_5D_Forward_LogReturn_Pct,2.435568,1.996392,-1.046904,-0.169177
3,ridge,Y1_ASPI_5D_Forward_LogReturn_Pct,2.950368,2.428042,-1.710664,-1.279462
4,svr,Y1_ASPI_5D_Forward_LogReturn_Pct,2.269350,1.779100,-0.539964,-0.054044
5,xgboost,Y1_ASPI_5D_Forward_LogReturn_Pct,3.044637,2.593328,-2.636664,-0.833791
6,gp,Y1_EventWindow_0_10_LogReturn_Pct,4.162126,3.716353,-2.662983,-0.003454
7,quantile,Y1_EventWindow_0_10_LogReturn_Pct,9.392762,7.687594,-20.300431,-3.715053
8,random_forest,Y1_EventWindow_0_10_LogReturn_Pct,4.505282,3.973258,-3.246010,-0.168591
9,ridge,Y1_EventWindow_0_10_LogReturn_Pct,4.715631,3.965050,-2.490291,-0.550337


## 4.6 Ablation: feature count, and stacking on the dense configuration

Two robustness checks on the choices above, both reusing the existing pipeline.

**Feature count.** Each fold trains on ~30 real rows (plus a handful of synthetic ones) while selecting K=20 features -- a high feature-to-row ratio that invites overfitting. K=10 is re-run here for comparison. **K=20 stays the primary configuration reported in Section 10 even if K=10 scores better below**: picking K by held-out performance would be selecting a hyperparameter on the test folds, which is exactly the kind of quiet optimism this notebook has been removing. This is reported as evidence about sensitivity, not as a new default.

**Stacking on the dense config.** The primary config loses 1 of 3 folds to the stacking warm-up. On the denser 20/5/5 config only 1 of 8 folds is lost, so the meta-learner is evaluated on far more real points. Members are RF+XGBoost only here, since the MLP was run on the primary splits only.

In [9]:
results_k10, _, _ = run_walk_forward(
    splits, X, y, dataset["event_date"], gdp_all=dataset["gdp_current_usd"],
    k_features=10, verbose=False)

print("Pooled R2, primary config, K=20 (reported) vs K=10 (ablation only):\n")
for name in ("ridge", "random_forest", "xgboost"):
    for t in TARGET_COLS:
        k20 = results[name][t]["pooled_r2"]
        k10 = results_k10[name][t]["pooled_r2"]
        rmse20 = np.mean(results[name][t]["rmse"])
        rmse10 = np.mean(results_k10[name][t]["rmse"])
        print(f"{name:>14s} | {t:<20s} K=20: R2={k20:+.4f} RMSE={rmse20:.4f}   "
              f"K=10: R2={k10:+.4f} RMSE={rmse10:.4f}")

print()
print("Stacking on the dense 20/5/5 config (RF+XGBoost; more folds means fewer test points lost):")
results_dense["stacked_dense"] = run_stack(
    results_dense, members=("random_forest", "xgboost"), label="stacked_dense",
)


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Pooled R2, primary config, K=20 (reported) vs K=10 (ablation only):

         ridge | Y1_ASPI_5D_Forward_LogReturn_Pct K=20: R2=-0.0421 RMSE=2.3118   K=10: R2=-0.0282 RMSE=2.2976
         ridge | Y2_abnormal_volume   K=20: R2=-0.4742 RMSE=0.6480   K=10: R2=-0.0061 RMSE=0.5174
         ridge | Y3_recovery_days     K=20: R2=-0.2020 RMSE=30.1026   K=10: R2=-0.2007 RMSE=30.2357
         ridge | Y1_EventWindow_0_5_LogReturn_Pct K=20: R2=+0.0543 RMSE=2.8156   K=10: R2=+0.0715 RMSE=2.8089
         ridge | Y1_EventWindow_0_10_LogReturn_Pct K=20: R2=+0.0071 RMSE=4.3934   K=10: R2=-0.0642 RMSE=4.5547
 random_forest | Y1_ASPI_5D_Forward_LogReturn_Pct K=20: R2=+0.0416 RMSE=2.2691   K=10: R2=+0.0532 RMSE=2.2835
 random_forest | Y2_abnormal_volume   K=20: R2=+0.2270 RMSE=0.4788   K=10: R2=+0.1348 RMSE=0.5058
 random_forest | Y3_recovery_days     K=20: R2=-0.1826 RMSE=29.7618   K=10: R2=-0.1964 RMSE=29.8796
 random_forest | Y1_EventWindow_0_5_LogReturn_Pct K=20: R2=+0.0458 RMSE=2.9248   K=10: R2=+0.1

## 4.7 Final models for the explainability stage

`results` holds the per-fold `y_true` / `y_pred` arrays for every model and target. Once
cached, the entire evaluation stage — bootstrap intervals, stratified error, ROC curves,
calibration, all the figures — is a pure function of this file and needs no re-fitting.

In [10]:
final_rf_models = {}
train_fit_r2 = {}
print("TRAIN-SET FIT R2 (in-sample -- RF refit on ALL real rows, scored on those SAME rows;")
print("measures memorization, NOT held-out predictive power. Compare only against the pooled")
print('R2 in the table above -- never report this number alone as "model accuracy".)')
print()
for t in TARGET_COLS:
    feat_cols = selected_features[t]
    ok = y[t].notna()
    rf_final = RandomForestRegressor(**last_models["random_forest"][t].get_params())
    rf_final.fit(X.loc[ok, feat_cols], y.loc[ok, t])
    final_rf_models[t] = rf_final
    in_sample_r2 = r2_score(y.loc[ok, t], clip_to_bounds(t, rf_final.predict(X.loc[ok, feat_cols])))
    train_fit_r2[t] = float(in_sample_r2)
    held_out = results["random_forest"][t]["pooled_r2"]
    print(f"random_forest | {t:<20s} train-fit R2={in_sample_r2:.4f}  (n={int(ok.sum())}, in-sample)"
          f"   vs held-out R2={held_out:+.4f}   gap={in_sample_r2 - held_out:.4f}")


TRAIN-SET FIT R2 (in-sample -- RF refit on ALL real rows, scored on those SAME rows;
measures memorization, NOT held-out predictive power. Compare only against the pooled
R2 in the table above -- never report this number alone as "model accuracy".)



random_forest | Y1_ASPI_5D_Forward_LogReturn_Pct train-fit R2=0.6043  (n=74, in-sample)   vs held-out R2=+0.0416   gap=0.5627


random_forest | Y2_abnormal_volume   train-fit R2=0.8495  (n=61, in-sample)   vs held-out R2=+0.2270   gap=0.6225


random_forest | Y3_recovery_days     train-fit R2=0.4650  (n=74, in-sample)   vs held-out R2=-0.1826   gap=0.6476


random_forest | Y1_EventWindow_0_5_LogReturn_Pct train-fit R2=0.8406  (n=74, in-sample)   vs held-out R2=+0.0458   gap=0.7948
random_forest | Y1_EventWindow_0_10_LogReturn_Pct train-fit R2=0.7208  (n=74, in-sample)   vs held-out R2=-0.0018   gap=0.7226


## 4.8 Ablation: does each external data block earn its place?

`docs/EXTERNAL_DATA_PRE_DECLARATION.md` commits to reporting this ablation whatever it
shows, and commits in advance to what is expected: the sample extension should help most
(it adds test points, not columns), hazard should help Y2/Y3 more than Y1, DesInventar
should act chiefly through `di_available` and `di_districts_hit`, FX should do little on
its own but is a correctness fix regardless, and **Y1 is still expected to be
unpredictable**.

Each row drops one block from the full feature set and refits on identical folds. A
block that helps shows a *positive* delta when removed hurts. Nothing here may revise
the pre-declared expectations after the fact.

In [11]:
from src.data_pipeline.external_sources import EXTERNAL_FEATURE_BLOCKS
from src.evaluation.verification import paired_bootstrap_delta

EXTERNAL_ALL = [c for cols in EXTERNAL_FEATURE_BLOCKS.values() for c in cols
                if c in FEATURE_COLS]
BASE_COLS = [c for c in FEATURE_COLS if c not in EXTERNAL_ALL]

# One configuration per row. "full" is the reported specification; "no_external" is the
# pre-external baseline; each "no_<block>" drops exactly one block from full.
ABLATION_SETS = {"full": FEATURE_COLS, "no_external": BASE_COLS}
for _blk, _cols in EXTERNAL_FEATURE_BLOCKS.items():
    _drop = set(_cols)
    ABLATION_SETS[f"no_{_blk}"] = [c for c in FEATURE_COLS if c not in _drop]

ablation_results = {}
for _name, _cols in ABLATION_SETS.items():
    _res, _, _ = run_walk_forward(
        splits, X[_cols], y, dataset["event_date"],
        gdp_all=dataset["gdp_current_usd"], k_features=20, verbose=False)
    ablation_results[_name] = _res
    print(f"  fitted {_name:16s} ({len(_cols):2d} features)")

rows = []
for _name, _res in ablation_results.items():
    for _model in ("ridge", "random_forest", "xgboost"):
        for _t in TARGET_COLS:
            _store = _res[_model][_t]
            # run_walk_forward records per-fold arrays; pooled R2 is computed by a
            # later cell for the primary results, so it is derived here rather than
            # read, otherwise this cell depends on cell ordering.
            _yt = np.concatenate(_store["y_true"])
            _yp = np.concatenate(_store["y_pred"])
            rows.append({"config": _name, "model": _model, "target": _t,
                         "pooled_r2": float(r2_score(_yt, _yp)),
                         "rmse": float(np.sqrt(np.mean((_yp - _yt) ** 2)))})
ablation_table = pd.DataFrame(rows)

print()
print("Pooled R2 by configuration (higher is better):")
print(ablation_table.pivot_table(index=["target", "model"], columns="config",
                                 values="pooled_r2").round(4).to_string())

# Paired event-level bootstrap of each block's contribution: full vs full-minus-block,
# on the same pooled out-of-fold points. A CI containing zero means the block is not
# distinguishable, and is reported as such.
print()
print("Block contribution -- Delta RMSE (full MINUS full-without-block); positive "
      "means the block helps. 95% CI from a paired event-level bootstrap:")
contrib = []
for _t in TARGET_COLS:
    for _model in ("ridge", "random_forest", "xgboost"):
        _full_t = np.concatenate(ablation_results["full"][_model][_t]["y_true"])
        _full_p = np.concatenate(ablation_results["full"][_model][_t]["y_pred"])
        for _blk in list(EXTERNAL_FEATURE_BLOCKS) + ["external"]:
            _key = "no_external" if _blk == "external" else f"no_{_blk}"
            _wo_p = np.concatenate(ablation_results[_key][_model][_t]["y_pred"])
            _d = paired_bootstrap_delta(_full_t, _full_p, _wo_p)
            contrib.append({"target": _t, "model": _model, "block": _blk,
                            "delta_rmse": _d["delta"],
                            "lo": _d["ci_low"], "hi": _d["ci_high"],
                            "significant": bool(_d["significant"])})
contrib_table = pd.DataFrame(contrib)
_sig = contrib_table[contrib_table.significant]
print(contrib_table.pivot_table(index=["target", "block"], columns="model",
                                values="delta_rmse").round(5).to_string())
print()
print(f"Blocks with a CI excluding zero: {len(_sig)} of {len(contrib_table)} "
      f"(model, target, block) combinations.")
if len(_sig):
    print(_sig[["target", "model", "block", "delta_rmse", "lo", "hi"]].to_string(index=False))
else:
    print("  None. No external block is distinguishable from noise on any target, "
          "at this N. Reported as measured.")

save_frame(ablation_table, "ablation_blocks", "Pooled R2/RMSE per external-block config.")
save_frame(contrib_table, "ablation_block_contrib",
           "Paired-bootstrap contribution of each external block.")

C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\Users\hasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


...

config                                             full  no_desinventar  no_election  no_external   no_fx  no_hazard
target                            model                                                                             
Y1_ASPI_5D_Forward_LogReturn_Pct  random_forest  0.0168         -0.0948       0.0024      -0.0141 -0.0509     0.0018
                                  ridge         -0.0569         -0.0440      -0.0328      -0.0206 -0.0361    -0.0239
                                  xgboost       -0.2902         -0.3044      -0.1574      -0.7830 -0.1823    -0.1026
Y1_EventWindow_0_10_LogReturn_Pct random_forest -0.0181         -0.1072      -0.0404      -0.0943 -0.0607    -0.0309
                                  ridge          0.0461         -0.0970      -0.0423      -0.0650 -0.0388    -0.0494
                                  xgboost       -0.1716         -0.1843      -0.1841      -0.1254 -0.3554    -0.1117
Y1_EventWindow_0_5_LogReturn_Pct  random_forest  0.0503         

model                                          random_forest    ridge  xgboost
target                            block                                       
Y1_ASPI_5D_Forward_LogReturn_Pct  desinventar        0.13305 -0.01528  0.01517
                                  election           0.01758 -0.02863 -0.14584
                                  external           0.03762 -0.04321  0.48446
                                  fx                 0.08166 -0.02463 -0.11788
                                  hazard             0.01837 -0.03931 -0.20842
Y1_EventWindow_0_10_LogReturn_Pct desinventar        0.21813  0.35693  0.02954
                                  election           0.05552  0.22359  0.02910
                                  external           0.18714  0.27942 -0.10876
                                  fx                 0.10550  0.21480  0.41298
                                  hazard             0.03195  0.24097 -0.14155
Y1_EventWindow_0_5_LogReturn_Pct  desinventar       

cached ablation_blocks.parquet  (90 rows x 5 cols)


cached ablation_block_contrib.parquet  (75 rows x 7 cols)


WindowsPath('F:/CSE-disaster-impact-predictor/artifacts/ablation_block_contrib.parquet')

## 4.8b Collinearity diagnostics (>0.75 pairs, reported not acted on) and a PCA ablation

The pre-declared redundancy rule (`src/evaluation/collinearity.py::redundant_drop_set`,
|rho|>=0.95) is now wired into `select_top_features` above and runs on every fold's
real training rows before RF-importance ranking. Separately, every pair correlated
above 0.75 is reported here as a diagnostic table -- surfaced for transparency, but the
actual drop threshold stays at the pre-registered 0.95 (dropping at 0.75 after seeing
results would not be a pre-declared rule any more).

PCA is not used anywhere in the pipeline. As an empirical check (not a proposal to
adopt it), this section compares Ridge on the RF-selected top-20 features against Ridge
on the top-N principal components of the SAME per-fold training rows -- same
train/test split, same purge, same scaling discipline, no leakage. PCA components are
uninterpretable, which cuts against the thesis's SHAP-based explainability chapter, so
this stays a reported comparison, not a replacement, unless it clearly wins.

In [12]:
from sklearn.decomposition import PCA
from src.evaluation.collinearity import top_correlated_pairs

corr_pairs = top_correlated_pairs(X, top_n=200)
corr_pairs_075 = corr_pairs[corr_pairs["abs_rho"] > 0.75].reset_index(drop=True)
print(f"{len(corr_pairs_075)} feature pairs with |rho| > 0.75 (diagnostic only -- drop "
      f"threshold stays at the pre-declared 0.95):")
print(corr_pairs_075.to_string(index=False))
save_frame(corr_pairs_075, "corr_pairs_above_075",
           "Diagnostic: feature pairs with |rho|>0.75 -- reported, never used to drop.")


def run_pca_ablation(n_components=10):
    """Ridge on top-N PCA components vs Ridge on the RF-selected top-20 raw features,
    same folds/purge/SMOGN as the main loop. PCA is fit on fold TRAIN rows only
    (StandardScaler + PCA in one pipeline, refit every fold -- no leakage)."""
    pca_results = {t: {"rmse": [], "mae": [], "r2": [], "y_true": [], "y_pred": []}
                   for t in TARGET_COLS}
    for fold_i, s in enumerate(splits):
        s = purge_horizon_overlap(s, dataset["event_date"], dataset["Y1_horizon_end_date"])
        X_tr_real, X_te = X.iloc[s.train_index], X.iloc[s.test_index]
        y_tr_real, y_te = y.iloc[s.train_index], y.iloc[s.test_index]
        dates_tr = dataset["event_date"].iloc[s.train_index]
        gdp_tr = dataset["gdp_current_usd"].iloc[s.train_index]
        X_tr_aug, y_tr_aug, _ = augment_fold(X_tr_real, y_tr_real, dates_tr, gdp_tr)

        for target in TARGET_COLS:
            use_log = target == "Y3_recovery_days"
            tr_ok, te_ok = y_tr_aug[target].notna(), y_te[target].notna()
            if int(y_tr_real[target].notna().sum()) < 10 or int(te_ok.sum()) < 1:
                continue
            y_tr_t, y_te_t = y_tr_aug.loc[tr_ok, target], y_te.loc[te_ok, target]
            y_tr_fit = np.log1p(y_tr_t) if use_log else y_tr_t
            n_comp = min(n_components, tr_ok.sum(), X.shape[1])
            pca_pipe = make_pipeline(
                StandardScaler(), PCA(n_components=n_comp, random_state=RANDOM_STATE),
                Ridge(alpha=1.0),
            )
            pca_pipe.fit(X_tr_aug.loc[tr_ok], y_tr_fit)
            pred = pca_pipe.predict(X_te.loc[te_ok])
            if use_log:
                pred = np.expm1(pred)
            pred = clip_to_bounds(target, pred)
            score_and_record(y_te_t, pred, pca_results[target])
    for t in TARGET_COLS:
        if pca_results[t]["y_true"]:
            yt_all = np.concatenate(pca_results[t]["y_true"])
            yp_all = np.concatenate(pca_results[t]["y_pred"])
            pca_results[t]["pooled_r2"] = float(r2_score(yt_all, yp_all))
    return pca_results


pca_results = run_pca_ablation(n_components=10)
print("\nPCA(10 components) + Ridge  vs  RF-selected top-20 features + Ridge (same folds):\n")
for t in TARGET_COLS:
    if "pooled_r2" not in pca_results[t]:
        continue
    pca_r2, pca_rmse = pca_results[t]["pooled_r2"], np.mean(pca_results[t]["rmse"])
    raw_r2, raw_rmse = results["ridge"][t]["pooled_r2"], np.mean(results["ridge"][t]["rmse"])
    print(f"{t:<20s} PCA: R2={pca_r2:+.4f} RMSE={pca_rmse:.4f}   "
          f"raw-selected: R2={raw_r2:+.4f} RMSE={raw_rmse:.4f}   "
          f"{'PCA better' if pca_r2 > raw_r2 else 'raw features better or equal'}")
save_object(pca_results, "pca_ablation_results",
            "PCA(10)+Ridge vs RF-selected top-20+Ridge, same folds -- reported comparison, not adopted by default.")

52 feature pairs with |rho| > 0.75 (diagnostic only -- drop threshold stays at the pre-declared 0.95):
              feature_a               feature_b       rho  abs_rho           group_a           group_b
          date_is_exact        disaster_Drought -1.000000 1.000000             other     disaster type
       financial_damage    log_financial_damage  1.000000 1.000000   damage/severity   damage/severity
    population_affected log_population_affected  1.000000 1.000000   damage/severity   damage/severity
            no_homeless      homeless_available  0.999218 0.999218             other             other
           mag_wind_kph      mag_wind_available  0.999218 0.999218             other             other
        price_to_sma_10         price_to_ema_10  0.980748 0.980748  returns/momentum  returns/momentum
           mag_area_km2      mag_area_available  0.977468 0.977468             other             other
   log_financial_damage           damage_to_gdp  0.970114 0.970114   dama

cached corr_pairs_above_075.parquet  (52 rows x 6 cols)



PCA(10 components) + Ridge  vs  RF-selected top-20 features + Ridge (same folds):

Y1_ASPI_5D_Forward_LogReturn_Pct PCA: R2=-0.4292 RMSE=2.6542   raw-selected: R2=-0.0421 RMSE=2.3118   raw features better or equal
Y2_abnormal_volume   PCA: R2=-0.0090 RMSE=0.5407   raw-selected: R2=-0.4742 RMSE=0.6480   PCA better
Y3_recovery_days     PCA: R2=-0.1729 RMSE=29.7683   raw-selected: R2=-0.2020 RMSE=30.1026   PCA better
Y1_EventWindow_0_5_LogReturn_Pct PCA: R2=-0.3852 RMSE=3.4099   raw-selected: R2=+0.0543 RMSE=2.8156   raw features better or equal
Y1_EventWindow_0_10_LogReturn_Pct PCA: R2=-0.6675 RMSE=5.7226   raw-selected: R2=+0.0071 RMSE=4.3934   raw features better or equal


cached pca_ablation_results.pkl


WindowsPath('F:/CSE-disaster-impact-predictor/artifacts/pca_ablation_results.pkl')

## 4.9 Cache every out-of-fold prediction

Stages 06 and 07 read these predictions rather than refitting anything. That is what
makes re-deriving a figure cheap — and what makes it impossible to accidentally tune
a model while re-plotting it.

In [13]:
save_object(results, "results_regression",
            f"Out-of-fold predictions, {len(splits)}-fold walk-forward, models: {sorted(results)}.")
save_object({"dense": results_dense, "k10": results_k10}, "results_ablations",
            "Split-geometry (20/5/5) and feature-count (K=10) ablations.")
save_object(selected_features, "selected_features", "Top-K features chosen per target, last fold.")
save_json(train_fit_r2, "train_fit_r2")
save_object(final_rf_models, "final_rf_models",
            "RandomForest refit on all real rows per target, for SHAP in stage 07.")


cached results_regression.pkl


cached results_ablations.pkl


cached selected_features.pkl
cached train_fit_r2.json


cached final_rf_models.pkl


WindowsPath('F:/CSE-disaster-impact-predictor/artifacts/final_rf_models.pkl')